# Stage 2: SPAM2020 crop-share model

## 1. このNotebookの目的

第2段階（作物選択・作物シェア）を推定するためのNotebookです。

今回は、第1段階の農地量を説明するための集約的なtop5変数を使わず、各グリッド・各作物のGAEZ climate potential raw yieldを使います。

削除した変数:

- rainfed_calorie_top5_raw
- irrigation_calorie_gain_top5_raw
- wx_50km_rainfed_calorie_top5_raw

## 2. モデルと目的変数

作物 c の効用を次のように置きます。

$$
U_{ic} = \alpha_c + \theta_Y \log(1 + Y^{climate}_{ic}) + X_i^\top \beta_c
$$

効用を作物シェアに変換します。

$$
s_{ic} = \frac{\exp(U_{ic})}{\sum_k \exp(U_{ik})}
$$

目的変数は、SPAM2020の全作物physical areaから作る条件付き作物シェアです。

$$
s^{SPAM}_{ic} = \frac{PA_{ic}}{\sum_k PA_{ik}}
$$

ここで PA はSPAMの作付面積（ha）です。したがって、まずはSPAMに存在する作物の構成比を説明します。

## 3. 重要な注意

GAEZの作物別climate potential raw yieldが、SPAMの作物コードと対応しているかを最初に確認します。対応しない作物をゼロで補完せず、対応表の確認が必要な場合はそこで停止します。

## 4. サーバー上のパスと設定


In [ ]:
from pathlib import Path
import gc
import json
import re
import warnings

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from scipy.optimize import minimize
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------
# Server paths
# ---------------------------------------------------------------------
ROOT = Path("/work/tsuda")
GAEZ_DIR = ROOT / "GAEZ"
HYDE_DIR = ROOT / "HYDE3.4"
CROPLAND_DIR = HYDE_DIR / "cropland_npys"
DIST_DIR = ROOT / "distance_to_cities"
GLOFAS_DIR = ROOT / "GloFAS" / "processed_5min"
FEATURE_CACHE = GAEZ_DIR / "CroplandRegression" / "features_cache"
LAND_MASK_DIR = GAEZ_DIR / "LandMasks"
DERIVED_MASK_DIR = LAND_MASK_DIR / "derived_5min"

SPAM_DIR = GAEZ_DIR / "SPAM2020"
CLIMATE_DIR = GAEZ_DIR / "ClimatePotential"
CLIMATE_CODE_TABLE = GAEZ_DIR / "GAEZ_RES02_climate_potential_crop_codes.csv"

OUTPUT_DIR = (
    GAEZ_DIR
    / "CroplandRegression"
    / "stage2_spam_crop_share_climate_potential_target2020"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2020
RANDOM_SEED = 42
N_SPLITS = 5
N_CROP_SAMPLE = 30_000
CROPLAND_THRESHOLD = 0.01
SHARE_EPS = 1e-6
REFERENCE_CROP = "RICE"
N_JOBS = 4
USE_SOIL_GROUP = True

# If automatic filename matching cannot find a GAEZ map, add the
# corresponding GAEZ filename token here, e.g. {"SPAM_CODE": "GAEZ_CODE"}.
CLIMATE_CODE_OVERRIDES = {}

print("SPAM_DIR:", SPAM_DIR)
print("CLIMATE_DIR:", CLIMATE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 5. 共通グリッド変数の読み込み


In [ ]:
def load_cache(*names):
    for name in names:
        path = FEATURE_CACHE / name
        if path.exists():
            return np.load(path, mmap_mode="r")
    raise FileNotFoundError(
        "None of the cache files exists:\n"
        + "\n".join(str(FEATURE_CACHE / n) for n in names)
    )


def safe_log1p(x):
    x = np.asarray(x, dtype=np.float32)
    return np.log1p(np.maximum(x, 0.0)).astype(np.float32)


def build_or_load_mode_cache(source_path, output_path, target_shape):
    if output_path.exists():
        arr = np.load(output_path, mmap_mode="r")
        if tuple(arr.shape) == tuple(target_shape):
            return arr

    if not source_path.exists():
        raise FileNotFoundError(source_path)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(source_path) as src:
        data = src.read(
            1,
            out_shape=target_shape,
            resampling=Resampling.mode,
            masked=True,
        )
        arr = data.filled(0).astype(np.int16)

    np.save(output_path, arr)
    return np.load(output_path, mmap_mode="r")


def take_grid(arr, rows, cols):
    return np.asarray(arr[rows, cols])


lat = np.load(CROPLAND_DIR / "lat.npy")
lon = np.load(CROPLAND_DIR / "lon.npy")
GRID_SHAPE = (len(lat), len(lon))
print("GRID_SHAPE:", GRID_SHAPE)

cropland_cube = np.load(
    CROPLAND_DIR / "cropland_fraction_1950_2024.npy",
    mmap_mode="r",
)
years = np.load(CROPLAND_DIR / "years.npy")
year_index = int(np.where(years == YEAR)[0][0])
cropland = np.asarray(cropland_cube[year_index], dtype=np.float32).copy()
cropland[~np.isfinite(cropland)] = np.nan

population = load_cache("population_density_2024.npy")
elevation = load_cache("elevation_5min.npy")
slope = load_cache("slope_5min.npy")

soil_group = build_or_load_mode_cache(
    LAND_MASK_DIR / "gaez_v5_wrb_soil_group_30sec.tif",
    DERIVED_MASK_DIR / "soil_group_5min_mode.npy",
    GRID_SHAPE,
)
exclusion = build_or_load_mode_cache(
    LAND_MASK_DIR / "gaez_v5_exclusion_30sec.tif",
    DERIVED_MASK_DIR / "exclusion_5min_mode.npy",
    GRID_SHAPE,
)

city_time = np.load(DIST_DIR / "cities_10_1_12deg_min.npy", mmap_mode="r")
port_time = np.load(DIST_DIR / "ports_05_1_12deg_min.npy", mmap_mode="r")
glofas = np.load(
    GLOFAS_DIR / "p10_discharge_max_5min_2020.npy",
    mmap_mode="r",
)
river_distance = np.load(
    GLOFAS_DIR / "distance_to_reliable_river_p10_gt_10_m3s_km_5min_2020.npy",
    mmap_mode="r",
)

# Variables retained for Stage 2:
# grid-level terrain/soil/access/water + crop-specific climate potential yield.
BASE_FEATURES = [
    "elevation_m",
    "slope",
    "exclusion_class",
    "log_city_time_20k_min",
    "log_port_time_any_min",
    "log_distance_river_gt10_2020",
    "log_glofas_p10_2020",
]
SOIL_FEATURES = BASE_FEATURES + ["soil_group_class"]
MODEL_FEATURES = SOIL_FEATURES if USE_SOIL_GROUP else BASE_FEATURES

CATEGORICAL_FEATURES = [
    f for f in ["exclusion_class", "soil_group_class"]
    if f in MODEL_FEATURES
]
NUMERIC_FEATURES = [
    f for f in MODEL_FEATURES
    if f not in CATEGORICAL_FEATURES
]

print("MODEL_FEATURES:")
print(MODEL_FEATURES)

## 6. 第2段階のグリッド説明変数


In [ ]:
def build_feature_frame(rows, cols):
    return pd.DataFrame({
        "row": rows.astype(np.int32),
        "col": cols.astype(np.int32),
        "lat": lat[rows].astype(np.float32),
        "lon": lon[cols].astype(np.float32),
        "cropland_fraction": take_grid(cropland, rows, cols).astype(np.float32),
        "elevation_m": take_grid(elevation, rows, cols).astype(np.float32),
        "slope": take_grid(slope, rows, cols).astype(np.float32),
        "exclusion_class": take_grid(exclusion, rows, cols).astype(np.int16),
        "soil_group_class": take_grid(soil_group, rows, cols).astype(np.int16),
        "log_city_time_20k_min": safe_log1p(take_grid(city_time, rows, cols)),
        "log_port_time_any_min": safe_log1p(take_grid(port_time, rows, cols)),
        "log_distance_river_gt10_2020": safe_log1p(
            take_grid(river_distance, rows, cols)
        ),
        "log_glofas_p10_2020": safe_log1p(take_grid(glofas, rows, cols)),
    })


land_mask = (
    np.isfinite(cropland)
    & (cropland > CROPLAND_THRESHOLD)
    & np.isfinite(np.asarray(population))
    & np.isfinite(np.asarray(elevation))
    & np.isfinite(np.asarray(slope))
)
print("Valid cropland cells:", int(land_mask.sum()))

## 7. SPAM2020の目的変数データ


In [ ]:
# ---------------------------------------------------------------------
# SPAM2020 crop-raster discovery
# ---------------------------------------------------------------------
SPAM_PATTERN = re.compile(
    r"_A_(?P<crop>[A-Z0-9]+)_(?P<system>[AIR])\.(?:tif|tiff)$",
    flags=re.IGNORECASE,
)

spam_files = sorted(
    p for p in SPAM_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".tif", ".tiff"}
)

spam_paths = {}
for path in spam_files:
    match = SPAM_PATTERN.search(path.name)
    if not match:
        continue
    crop = match.group("crop").upper()
    system = match.group("system").upper()
    spam_paths.setdefault(crop, {})[system] = path

if not spam_paths:
    raise FileNotFoundError(f"No SPAM rasters were found below {SPAM_DIR}")

crop_codes = sorted(spam_paths)
if REFERENCE_CROP not in crop_codes:
    raise ValueError(f"{REFERENCE_CROP} is not in SPAM crop codes")

print("SPAM crop count:", len(crop_codes))
print(crop_codes)

with rasterio.open(next(iter(spam_paths[REFERENCE_CROP].values()))) as src:
    spam_shape = (src.height, src.width)
    print("SPAM raster shape:", spam_shape)
    print("SPAM CRS:", src.crs)
    print("SPAM transform:", src.transform)

if spam_shape != GRID_SHAPE:
    raise ValueError(
        f"SPAM shape {spam_shape} != model grid shape {GRID_SHAPE}"
    )


def read_spam_area(crop):
    paths = spam_paths[crop]

    if "A" in paths:
        with rasterio.open(paths["A"]) as src:
            arr = src.read(1, masked=True).filled(0).astype(np.float32)
    else:
        arr = np.zeros(GRID_SHAPE, dtype=np.float32)
        for system in ("I", "R"):
            if system in paths:
                with rasterio.open(paths[system]) as src:
                    part = src.read(1, masked=True).filled(0).astype(np.float32)
                part[~np.isfinite(part)] = 0
                part[part < 0] = 0
                arr += part

    arr[~np.isfinite(arr)] = 0
    arr[arr < 0] = 0
    return arr

## 8. GAEZ作物別climate potential raw収量


In [ ]:
# ---------------------------------------------------------------------
# GAEZ climate-potential raw-yield map discovery
# ---------------------------------------------------------------------
def filename_has_token(path, token):
    stem = path.stem.upper()
    token = str(token).upper().strip()
    return re.search(
        rf"(?<![A-Z0-9]){re.escape(token)}(?![A-Z0-9])",
        stem,
    ) is not None


climate_files = sorted(
    p for p in CLIMATE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in {".tif", ".tiff", ".npy"}
)

if not climate_files:
    raise FileNotFoundError(
        f"No .tif, .tiff, or .npy files were found below {CLIMATE_DIR}"
    )

print("Candidate climate files:", len(climate_files))
print("Climate code table exists:", CLIMATE_CODE_TABLE.exists())

if CLIMATE_CODE_TABLE.exists():
    climate_code_table = pd.read_csv(CLIMATE_CODE_TABLE)
    print("Climate code table columns:", list(climate_code_table.columns))
    display(climate_code_table.head())


def resolve_climate_path(spam_code):
    alias = CLIMATE_CODE_OVERRIDES.get(spam_code, spam_code)
    candidates = [
        p for p in climate_files
        if filename_has_token(p, alias)
        and "TOP5" not in p.stem.upper()
    ]

    if len(candidates) == 0:
        return None
    if len(candidates) > 1:
        preferred = [
            p for p in candidates
            if any(word in p.stem.upper() for word in ["RAW", "YIELD", "POTENTIAL"])
        ]
        if len(preferred) == 1:
            return preferred[0]
        print(f"Ambiguous climate files for {spam_code}:")
        for p in candidates[:20]:
            print("  ", p)
        raise ValueError(
            f"Set CLIMATE_CODE_OVERRIDES or narrow the files for {spam_code}"
        )
    return candidates[0]


climate_paths = {
    crop: resolve_climate_path(crop)
    for crop in crop_codes
}
missing_climate = [crop for crop, path in climate_paths.items() if path is None]

print("Matched climate maps:", len(crop_codes) - len(missing_climate))
if missing_climate:
    print("Missing climate maps:", missing_climate)
    raise FileNotFoundError(
        "GAEZ climate-potential raw-yield maps could not be matched for "
        + ", ".join(missing_climate)
        + ". Add entries to CLIMATE_CODE_OVERRIDES after checking the printed files."
    )

display(pd.DataFrame({
    "spam_crop": crop_codes,
    "climate_map": [str(climate_paths[c]) for c in crop_codes],
}))

## 9. サンプル作成と作物シェア


In [ ]:
def read_climate_map(path):
    if path.suffix.lower() == ".npy":
        arr = np.asarray(np.load(path))
        if arr.ndim != 2:
            raise ValueError(
                f"{path} has shape {arr.shape}; expected one 2-D map per crop."
            )
        if tuple(arr.shape) != GRID_SHAPE:
            raise ValueError(
                f"{path} has shape {arr.shape}, expected {GRID_SHAPE}."
            )
    else:
        with rasterio.open(path) as src:
            if (src.height, src.width) == GRID_SHAPE:
                arr = src.read(1, masked=True).filled(np.nan)
            else:
                print(
                    f"Resampling {path.name}: "
                    f"{(src.height, src.width)} -> {GRID_SHAPE}"
                )
                arr = src.read(
                    1,
                    out_shape=GRID_SHAPE,
                    resampling=Resampling.bilinear,
                    masked=True,
                ).filled(np.nan)

    arr = np.asarray(arr, dtype=np.float32)
    arr[~np.isfinite(arr)] = np.nan
    arr[arr < 0] = np.nan
    return arr


# Read the total SPAM area for the sampling denominator.
total_spam_area = np.zeros(GRID_SHAPE, dtype=np.float32)
for i, crop in enumerate(crop_codes, start=1):
    area = read_spam_area(crop)
    total_spam_area += area
    del area
    if i % 10 == 0 or i == len(crop_codes):
        print(f"{i}/{len(crop_codes)} SPAM total maps processed")

candidate_flat = np.flatnonzero(
    land_mask.ravel()
    & (total_spam_area.ravel() > 0)
)
if len(candidate_flat) == 0:
    raise ValueError("No valid cells have positive SPAM area.")

rng = np.random.default_rng(RANDOM_SEED)
n_sample = min(N_CROP_SAMPLE, len(candidate_flat))
selected_flat = np.sort(
    rng.choice(candidate_flat, size=n_sample, replace=False)
)
rows, cols = np.unravel_index(selected_flat, GRID_SHAPE)

area_sample = np.zeros((n_sample, len(crop_codes)), dtype=np.float32)
climate_raw_sample = np.zeros((n_sample, len(crop_codes)), dtype=np.float32)

for j, crop in enumerate(crop_codes):
    area = read_spam_area(crop)
    area_sample[:, j] = area[rows, cols]
    del area

    climate_map = read_climate_map(climate_paths[crop])
    climate_raw_sample[:, j] = climate_map[rows, cols]
    del climate_map

total_area_sample = area_sample.sum(axis=1)
valid = np.isfinite(total_area_sample) & (total_area_sample > 0)
valid &= np.isfinite(climate_raw_sample).all(axis=1)

rows = rows[valid]
cols = cols[valid]
area_sample = area_sample[valid]
climate_raw_sample = climate_raw_sample[valid]
total_area_sample = total_area_sample[valid]

y_share = area_sample / total_area_sample[:, None]
y_share = np.nan_to_num(y_share, nan=0.0, posinf=0.0, neginf=0.0)
y_share = y_share / y_share.sum(axis=1, keepdims=True)

sample = build_feature_frame(rows, cols)
sample["total_spam_area_ha"] = total_area_sample.astype(np.float32)

grid_area_path = GAEZ_DIR / "grid_area_2160x4320_ha.npy"
if grid_area_path.exists():
    grid_area_ha = np.load(grid_area_path, mmap_mode="r")
else:
    grid_area_km2_path = CROPLAND_DIR / "grid_area_km2.npy"
    grid_area_ha = np.load(grid_area_km2_path, mmap_mode="r") * 100.0

cropland_area_sample = (
    grid_area_ha[rows, cols].astype(np.float32)
    * sample["cropland_fraction"].to_numpy(dtype=np.float32)
)
sample["spam_coverage"] = (
    sample["total_spam_area_ha"].to_numpy(dtype=np.float32)
    / np.maximum(cropland_area_sample, 1e-6)
).astype(np.float32)

valid_features = np.isfinite(
    sample[NUMERIC_FEATURES].to_numpy(dtype=np.float32)
).all(axis=1)
valid_features &= sample[CATEGORICAL_FEATURES].notna().all(axis=1).to_numpy()

sample = sample.loc[valid_features].reset_index(drop=True)
y_share = y_share[valid_features]
climate_raw_sample = climate_raw_sample[valid_features]
area_sample = area_sample[valid_features]

sample["spatial_block"] = (
    np.floor((sample["lat"] + 90.0) / 10.0).astype(np.int32) * 36
    + np.floor((sample["lon"] + 180.0) / 10.0).astype(np.int32)
)

sample_weight = sample["total_spam_area_ha"].to_numpy(dtype=np.float64)
sample_weight = sample_weight / np.maximum(np.nanmean(sample_weight), 1e-12)

climate_log_sample = np.log1p(
    np.maximum(climate_raw_sample, 0.0)
).astype(np.float32)

np.save(
    OUTPUT_DIR / "stage2_climate_potential_raw_yield_sample.npy",
    climate_raw_sample,
)
print("Final sample:", sample.shape)
print("Climate target matrix:", climate_raw_sample.shape)
print("Share-sum range:", y_share.sum(axis=1).min(), y_share.sum(axis=1).max())
print("SPAM coverage quantiles:")
print(sample["spam_coverage"].quantile([0, .01, .05, .5, .95, .99, 1.0]))

## 10. モデルの考え方

線形モデルでは、作物別climate potential yieldはグリッドごとに値が異なるため、作物選択の自然的比較優位として効きます。

地形・土壌・市場アクセスなどのグリッド変数は、作物別係数を持たせます。

非線形モデルでは、グリッドを作物数だけlong形式に展開し、crop_codeと作物別climate potential yieldをLightGBMに入力します。


## 11. 線形softmaxモデル


In [ ]:
def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_grid_design(train_frame, test_frame):
    scaler = StandardScaler()
    x_num_train = scaler.fit_transform(
        train_frame[NUMERIC_FEATURES].to_numpy(dtype=np.float64)
    )
    x_num_test = scaler.transform(
        test_frame[NUMERIC_FEATURES].to_numpy(dtype=np.float64)
    )

    encoder = None
    if CATEGORICAL_FEATURES:
        encoder = make_encoder()
        x_cat_train = encoder.fit_transform(
            train_frame[CATEGORICAL_FEATURES].astype(str)
        )
        x_cat_test = encoder.transform(
            test_frame[CATEGORICAL_FEATURES].astype(str)
        )
        cat_names = list(
            encoder.get_feature_names_out(CATEGORICAL_FEATURES)
        )
    else:
        x_cat_train = np.empty((len(train_frame), 0), dtype=np.float64)
        x_cat_test = np.empty((len(test_frame), 0), dtype=np.float64)
        cat_names = []

    x_train = np.column_stack([x_num_train, x_cat_train]).astype(np.float64)
    x_test = np.column_stack([x_num_test, x_cat_test]).astype(np.float64)
    return (
        x_train,
        x_test,
        NUMERIC_FEATURES + cat_names,
        scaler,
        encoder,
    )


def standardize_climate(train_log, test_log):
    mean = np.nanmean(train_log, axis=0)
    std = np.nanstd(train_log, axis=0)
    std = np.where(std < 1e-8, 1.0, std)
    return (
        (train_log - mean[None, :]) / std[None, :],
        (test_log - mean[None, :]) / std[None, :],
        mean,
        std,
    )


def stable_softmax(scores):
    scores = np.asarray(scores, dtype=np.float64)
    shifted = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(np.clip(shifted, -50, 50))
    return exp_scores / np.maximum(exp_scores.sum(axis=1, keepdims=True), 1e-300)


def fit_linear_softmax(
    x_grid,
    climate_z,
    y,
    weights,
    crop_codes,
    reference_crop=REFERENCE_CROP,
    l2=0.02,
    maxiter=200,
):
    n, p = x_grid.shape
    k = len(crop_codes)
    ref_idx = crop_codes.index(reference_crop)
    non_ref = np.asarray(
        [j for j in range(k) if j != ref_idx],
        dtype=np.int32,
    )
    x_aug = np.column_stack([np.ones(n), x_grid])
    w = np.asarray(weights, dtype=np.float64)
    w = w / np.maximum(w.mean(), 1e-12)
    y = np.asarray(y, dtype=np.float64)
    climate_z = np.asarray(climate_z, dtype=np.float64)

    def objective_gradient(theta):
        beta = theta[:-1].reshape(k - 1, p + 1)
        climate_beta = theta[-1]
        scores = climate_beta * climate_z
        scores[:, non_ref] += x_aug @ beta.T
        probs = stable_softmax(scores)

        loss = -np.sum(
            w[:, None] * y * np.log(np.clip(probs, 1e-12, 1.0))
        ) / np.sum(w)
        loss += 0.5 * l2 * np.sum(beta[:, 1:] ** 2)
        loss += 0.5 * l2 * climate_beta ** 2

        score_gradient = (w[:, None] * (probs - y)) / np.sum(w)
        beta_gradient = score_gradient[:, non_ref].T @ x_aug
        beta_gradient[:, 1:] += l2 * beta[:, 1:]
        climate_gradient = np.sum(score_gradient * climate_z)
        climate_gradient += l2 * climate_beta

        return float(loss), np.r_[beta_gradient.ravel(), climate_gradient]

    initial = np.zeros((k - 1) * (p + 1) + 1, dtype=np.float64)
    result = minimize(
        fun=lambda theta: objective_gradient(theta)[0],
        x0=initial,
        jac=lambda theta: objective_gradient(theta)[1],
        method="L-BFGS-B",
        options={"maxiter": maxiter, "ftol": 1e-8},
    )

    return {
        "beta": result.x[:-1].reshape(k - 1, p + 1),
        "climate_beta": result.x[-1],
        "non_ref": non_ref,
        "reference_crop": reference_crop,
        "crop_codes": list(crop_codes),
        "optimizer_result": result,
    }


def predict_linear_softmax(model, x_grid, climate_z):
    n = len(x_grid)
    x_aug = np.column_stack([np.ones(n), x_grid])
    scores = model["climate_beta"] * np.asarray(climate_z)
    scores[:, model["non_ref"]] += x_aug @ model["beta"].T
    return stable_softmax(scores)

## 12. 非線形LightGBMモデル


In [ ]:
LONG_CLIMATE_FEATURE = "log_climate_potential_raw_yield"
LONG_FEATURES = MODEL_FEATURES + [LONG_CLIMATE_FEATURE, "crop_code"]


def make_long_frame(frame, climate_log, y=None, weights=None):
    n = len(frame)
    k = len(crop_codes)
    repeated_index = np.repeat(np.arange(n), k)

    long_frame = frame.iloc[repeated_index][MODEL_FEATURES].reset_index(drop=True).copy()
    long_frame[LONG_CLIMATE_FEATURE] = np.asarray(
        climate_log,
        dtype=np.float32,
    ).reshape(-1)
    long_frame["crop_code"] = pd.Categorical(
        np.tile(np.asarray(crop_codes, dtype=object), n),
        categories=crop_codes,
    )

    for col in CATEGORICAL_FEATURES:
        if col == "exclusion_class":
            levels = list(range(0, 20))
        else:
            levels = list(range(0, 100))
        long_frame[col] = pd.Categorical(
            long_frame[col],
            categories=levels,
        )

    if y is not None:
        long_frame["target_log_share"] = np.log(
            np.asarray(y, dtype=np.float64).reshape(-1) + SHARE_EPS
        )
    if weights is not None:
        long_frame["sample_weight"] = np.repeat(
            np.asarray(weights, dtype=np.float64),
            k,
        )
    return long_frame


def fit_nonlinear_score_model(frame, climate_log, y, weights):
    long_train = make_long_frame(
        frame,
        climate_log,
        y=y,
        weights=weights,
    )
    categorical_for_lgbm = ["crop_code"] + CATEGORICAL_FEATURES

    model = LGBMRegressor(
        objective="regression",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=80,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        verbosity=-1,
    )
    model.fit(
        long_train[LONG_FEATURES],
        long_train["target_log_share"],
        sample_weight=long_train["sample_weight"],
        categorical_feature=categorical_for_lgbm,
    )
    del long_train
    gc.collect()
    return model


def predict_nonlinear_score_model(model, frame, climate_log):
    long_test = make_long_frame(frame, climate_log)
    score = model.predict(long_test[LONG_FEATURES])
    score = np.asarray(score, dtype=np.float64).reshape(
        len(frame),
        len(crop_codes),
    )
    del long_test
    return stable_softmax(score)

## 13. 評価指標


In [ ]:
def weighted_cross_entropy(y_true, y_pred, weights):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    w = np.asarray(weights, dtype=np.float64)
    return float(
        -np.sum(w[:, None] * y_true * np.log(np.clip(y_pred, 1e-12, 1.0)))
        / np.sum(w)
    )


def weighted_share_rmse(y_true, y_pred, weights):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    w = np.asarray(weights, dtype=np.float64)
    row_mse = np.mean((y_true - y_pred) ** 2, axis=1)
    return float(np.sqrt(np.sum(w * row_mse) / np.sum(w)))


def dominant_accuracy(y_true, y_pred, weights):
    true_idx = np.argmax(y_true, axis=1)
    pred_idx = np.argmax(y_pred, axis=1)
    return float(
        np.average((true_idx == pred_idx).astype(float), weights=weights)
    )


def evaluate(name, y_true, y_pred, weights):
    return {
        "model": name,
        "weighted_cross_entropy": weighted_cross_entropy(
            y_true, y_pred, weights
        ),
        "weighted_share_rmse": weighted_share_rmse(
            y_true, y_pred, weights
        ),
        "dominant_crop_accuracy": dominant_accuracy(
            y_true, y_pred, weights
        ),
    }

## 14. 空間ブロックOOF評価


In [ ]:
# ---------------------------------------------------------------------
# Spatial OOF comparison
# ---------------------------------------------------------------------
groups = sample["spatial_block"].to_numpy()
splitter = GroupKFold(n_splits=N_SPLITS)

linear_oof = np.full_like(y_share, np.nan, dtype=np.float32)
nonlinear_oof = np.full_like(y_share, np.nan, dtype=np.float32)
fold_metrics = []

for fold_number, (train_idx, test_idx) in enumerate(
    splitter.split(sample, groups=groups),
    start=1,
):
    print(f"--- fold {fold_number}/{N_SPLITS} ---")

    train_frame = sample.iloc[train_idx].copy()
    test_frame = sample.iloc[test_idx].copy()

    y_train = y_share[train_idx]
    y_test = y_share[test_idx]
    w_train = sample_weight[train_idx]
    w_test = sample_weight[test_idx]

    climate_train_z, climate_test_z, _, _ = standardize_climate(
        climate_log_sample[train_idx],
        climate_log_sample[test_idx],
    )

    x_train, x_test, design_names, scaler, encoder = make_grid_design(
        train_frame,
        test_frame,
    )

    linear_model = fit_linear_softmax(
        x_train,
        climate_train_z,
        y_train,
        w_train,
        crop_codes,
        reference_crop=REFERENCE_CROP,
        l2=0.02,
        maxiter=200,
    )
    pred_linear = predict_linear_softmax(
        linear_model,
        x_test,
        climate_test_z,
    )
    linear_oof[test_idx] = pred_linear.astype(np.float32)

    nonlinear_model = fit_nonlinear_score_model(
        train_frame,
        climate_log_sample[train_idx],
        y_train,
        w_train,
    )
    pred_nonlinear = predict_nonlinear_score_model(
        nonlinear_model,
        test_frame,
        climate_log_sample[test_idx],
    )
    nonlinear_oof[test_idx] = pred_nonlinear.astype(np.float32)

    fold_metrics.append({
        "fold": fold_number,
        "n_train": len(train_idx),
        "n_test": len(test_idx),
        **{
            f"linear_{key}": value
            for key, value in evaluate(
                "linear",
                y_test,
                pred_linear,
                w_test,
            ).items()
            if key != "model"
        },
        **{
            f"nonlinear_{key}": value
            for key, value in evaluate(
                "nonlinear",
                y_test,
                pred_nonlinear,
                w_test,
            ).items()
            if key != "model"
        },
    })

    del (
        train_frame,
        test_frame,
        x_train,
        x_test,
        linear_model,
        nonlinear_model,
        pred_linear,
        pred_nonlinear,
    )
    gc.collect()

metrics_df = pd.DataFrame(fold_metrics)
display(metrics_df)
display(metrics_df.mean(numeric_only=True).to_frame("mean").T)

metrics_df.to_csv(
    OUTPUT_DIR / "stage2_climate_potential_spatial_oof_metrics.csv",
    index=False,
)
np.save(
    OUTPUT_DIR / "stage2_climate_potential_linear_oof.npy",
    linear_oof,
)
np.save(
    OUTPUT_DIR / "stage2_climate_potential_nonlinear_oof.npy",
    nonlinear_oof,
)
print("OOF outputs saved.")

## 15. OOF結果の集計


In [ ]:
# Overall metrics and target diagnostics.
global_share = np.average(y_share, axis=0, weights=sample_weight)
global_pred = np.broadcast_to(global_share[None, :], y_share.shape)

overall_metrics = pd.DataFrame([
    evaluate(
        "global_share_baseline",
        y_share,
        global_pred,
        sample_weight,
    ),
    evaluate(
        "linear_fractional_softmax",
        y_share,
        linear_oof,
        sample_weight,
    ),
    evaluate(
        "nonlinear_lightgbm_score_softmax",
        y_share,
        nonlinear_oof,
        sample_weight,
    ),
])
display(overall_metrics)

overall_metrics.to_csv(
    OUTPUT_DIR / "stage2_climate_potential_overall_oof_metrics.csv",
    index=False,
)

dominant_index = np.argmax(y_share, axis=1)
diagnostic = sample[
    [
        "row",
        "col",
        "lat",
        "lon",
        "cropland_fraction",
        "total_spam_area_ha",
        "spam_coverage",
        "spatial_block",
    ]
].copy()
diagnostic["dominant_crop"] = np.asarray(crop_codes, dtype=object)[
    dominant_index
]
diagnostic["dominant_share"] = y_share[
    np.arange(len(y_share)),
    dominant_index,
].astype(np.float32)
diagnostic.to_csv(
    OUTPUT_DIR / "stage2_climate_potential_sample_diagnostics.csv.gz",
    index=False,
    compression="gzip",
)

## 16. 線形モデルの係数保存


In [ ]:
# Full-sample linear coefficients.
x_all, _, design_names, scaler_all, encoder_all = make_grid_design(
    sample,
    sample,
)
climate_all_z, _, climate_mean, climate_std = standardize_climate(
    climate_log_sample,
    climate_log_sample,
)
linear_full = fit_linear_softmax(
    x_all,
    climate_all_z,
    y_share,
    sample_weight,
    crop_codes,
    reference_crop=REFERENCE_CROP,
    l2=0.02,
    maxiter=250,
)

if encoder_all is not None:
    full_design_names = NUMERIC_FEATURES + list(
        encoder_all.get_feature_names_out(CATEGORICAL_FEATURES)
    )
else:
    full_design_names = list(NUMERIC_FEATURES)

coef_names = ["intercept"] + full_design_names
coef_table = pd.DataFrame(
    linear_full["beta"],
    index=[c for c in crop_codes if c != REFERENCE_CROP],
    columns=coef_names,
)
coef_table.index.name = "crop_relative_to_reference"
coef_table.to_csv(
    OUTPUT_DIR / "stage2_linear_relative_utility_coefficients.csv"
)

pd.DataFrame({
    "term": ["log_climate_potential_raw_yield_standardized"],
    "coefficient": [linear_full["climate_beta"]],
}).to_csv(
    OUTPUT_DIR / "stage2_linear_climate_potential_coefficient.csv",
    index=False,
)

display(coef_table.head())
print("Reference crop:", REFERENCE_CROP)

## 17. 非線形モデルのSHAP分析


In [ ]:
# Optional SHAP summary for nonlinear log-share scores.
try:
    import shap

    nonlinear_full = fit_nonlinear_score_model(
        sample,
        climate_log_sample,
        y_share,
        sample_weight,
    )
    long_for_shap = make_long_frame(sample, climate_log_sample)
    shap_n = min(20_000, len(long_for_shap))
    shap_rng = np.random.default_rng(RANDOM_SEED + 1000)
    shap_idx = np.sort(
        shap_rng.choice(len(long_for_shap), size=shap_n, replace=False)
    )
    shap_frame = long_for_shap.iloc[shap_idx][LONG_FEATURES]

    explainer = shap.TreeExplainer(nonlinear_full)
    shap_values = explainer.shap_values(shap_frame)
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    shap_values = np.asarray(shap_values)

    shap_summary = pd.DataFrame({
        "feature": LONG_FEATURES,
        "mean_abs_shap_log_share": np.nanmean(
            np.abs(shap_values),
            axis=0,
        ),
        "mean_signed_shap_log_share": np.nanmean(
            shap_values,
            axis=0,
        ),
    }).sort_values(
        "mean_abs_shap_log_share",
        ascending=False,
    )

    feature_group = {
        "elevation_m": "land_soil",
        "slope": "land_soil",
        "soil_group_class": "land_soil",
        "exclusion_class": "land_soil",
        "log_city_time_20k_min": "market_access",
        "log_port_time_any_min": "market_access",
        "log_distance_river_gt10_2020": "water_access",
        "log_glofas_p10_2020": "water_access",
        "log_climate_potential_raw_yield": "crop_specific_climate_suitability",
        "crop_code": "crop_identity_control",
    }
    shap_summary["group"] = shap_summary["feature"].map(feature_group)
    shap_summary.to_csv(
        OUTPUT_DIR / "stage2_nonlinear_shap_feature_summary.csv",
        index=False,
    )

    group_summary = (
        shap_summary[
            shap_summary["group"] != "crop_identity_control"
        ]
        .groupby("group", as_index=False)["mean_abs_shap_log_share"]
        .sum()
        .sort_values(
            "mean_abs_shap_log_share",
            ascending=False,
        )
    )
    group_summary.to_csv(
        OUTPUT_DIR / "stage2_nonlinear_shap_group_summary.csv",
        index=False,
    )
    display(shap_summary)
    display(group_summary)
except ImportError:
    print("shap is not installed; skipped SHAP summary.")

## 18. 出力と次の段階


In [ ]:
metadata = {
    "year": YEAR,
    "spam_dir": str(SPAM_DIR),
    "climate_dir": str(CLIMATE_DIR),
    "n_crops": len(crop_codes),
    "crop_codes": crop_codes,
    "target": "SPAM conditional crop share",
    "target_formula": "PA_ic / sum_k(PA_ik)",
    "removed_features": [
        "rainfed_calorie_top5_raw",
        "irrigation_calorie_gain_top5_raw",
        "wx_50km_rainfed_calorie_top5_raw",
    ],
    "model_features": MODEL_FEATURES,
    "crop_specific_feature": "log1p(GAEZ climate potential raw yield)",
    "categorical_features": CATEGORICAL_FEATURES,
    "reference_crop": REFERENCE_CROP,
    "sample_size": int(len(sample)),
    "spatial_block": "10-degree latitude/longitude blocks",
    "linear_model": "fractional cross-entropy softmax",
    "nonlinear_model": "pooled LightGBM log-share score followed by softmax",
}
with open(
    OUTPUT_DIR / "stage2_climate_potential_model_metadata.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("All outputs:", OUTPUT_DIR)
print("The raw climate-potential sample was saved separately.")